# Day 3 — Hands-On Lab 2: ADLS → Autoloader → Bronze (Customers + Payments)

**GlobalMart Data Engineering Bootcamp**

| | |
|---|---|
| **Datasets** | `customers`, `payments` — CSV files under `raw-data/` in ADLS |
| **Storage** | Unity Catalog External Location `gbmart-ext-loc` → `ecomadlsdata` — no storage keys, auth is automatic |
| **Time** | 3:00 PM – 4:00 PM (60 minutes) |
| **Layer** | Bronze — managed Delta tables in `gbmart.bronze` |
| **Pipeline role** | This IS the real GlobalMart pipeline — `customers` and `payments` are two of the four entities that arrive via ADLS file drop + Autoloader (the other two are `products` and `address`). `orders`/`order_items` arrive separately via Postgres CDC (Day 2) — not covered here. |

**Goal:** Use Databricks Autoloader to read `customers` and `payments` files from ADLS
and register them in Bronze as managed Delta tables — `gbmart.bronze.customers` and `gbmart.bronze.payments`.

**What makes Autoloader special:**
- It **automatically detects new files** dropped in the ADLS folder
- It **remembers which files it already processed** (no duplicates)
- It **infers schema** automatically — you don't have to define columns
- It is built for **incremental ingestion** — production-grade from day one

---
## What is Autoloader?

```
Without Autoloader (manual approach):
  You read all files every time → expensive
  Or you manually track which files are new → complex code

With Autoloader:
  Run 1: ADLS has customers_010626.csv  → Autoloader reads it, saves checkpoint
  Run 2: ADLS has customers_020626.csv  → Autoloader reads ONLY the new file
  Run 3: No new files                   → Autoloader does nothing (very fast)
```

**Autoloader = incremental file ingestion made easy.**

---
**Instructions:** Run each cell with **Shift + Enter**. This lab follows directly from ILT 2 (Autoloader & Schema Evolution Concepts, 2:00–3:00 PM).

---
## Key Autoloader Concepts

| Term | What it means |
|------|---------------|
| `cloudFiles` | The Autoloader format — like saying `format("delta")` but for reading files incrementally |
| `cloudFiles.format` | What kind of files you're reading — `csv`, `json`, `parquet` |
| `cloudFiles.schemaLocation` | Where Autoloader stores the inferred schema — so it's consistent across runs |
| `checkpointLocation` | Where Autoloader tracks which files have been processed — the "memory" |
| `trigger(availableNow=True)` | Process all available new files and stop — like a batch job |
| `trigger(processingTime='1 minute')` | Process new files every 1 minute — like a streaming job |
| `.toTable(...)` | Registers the stream's output as a real Unity Catalog managed table, not just files at a path |

---
## Pre-requisites

```
Make sure these folders exist under the GlobalMart external location:
  raw-data/customers/   ✅
  raw-data/payments/    ✅

Autoloader does NOT need the JDBC driver, and does NOT need a storage key.
Unity Catalog's External Location already grants this cluster read access —
that's the whole point of the storage credential you set up in Day 2 HOL 1.
```

In [ ]:
# ─── Setup: Unity Catalog External Location — NO storage keys needed ──────────
# The GlobalMart external location was already created in Unity Catalog on
# Day 2 (Day2_HOL1_Lakeflow_Connect_Storage_Credentials). Any cluster with
# access to the gbmart catalog can read this path — there is no
# spark.conf.set() and no storage account key anywhere in this notebook.

EXTERNAL_LOCATION = "abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data"
CATALOG           = "gbmart"
SCHEMA            = "bronze"

# Make sure the target catalog/schema exist before we write to them
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Autoloader needs two folders PER SOURCE to track its own state:
#   _schemas/<table>/     — the schema Autoloader inferred (reused across runs)
#   _checkpoints/<table>/ — which files have already been processed ("memory")
print("Connected via Unity Catalog External Location — no keys required.")
print(f"Source root : {EXTERNAL_LOCATION}")
print(f"Target      : {CATALOG}.{SCHEMA}.<table>")

---
## Part 1 — Ingest customers.csv with Autoloader

In [ ]:
# ─── Step 1: Read customers/ using Autoloader (cloudFiles) ────────────────────
# This does NOT read any data yet — it only defines the stream.
# cloudFiles.inferColumnTypes=true gives real types (not everything-as-string).

customers_source_path     = f"{EXTERNAL_LOCATION}/customers/"
customers_schema_path     = f"{EXTERNAL_LOCATION}/_schemas/customers/"
customers_checkpoint_path = f"{EXTERNAL_LOCATION}/_checkpoints/customers/"
customers_target_table    = f"{CATALOG}.{SCHEMA}.customers"

from pyspark.sql.functions import current_timestamp, col

customers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",           "csv")
    .option("cloudFiles.schemaLocation",   customers_schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header",                      "true")
    .load(customers_source_path)
    # Audit columns: which file each row came from, and when we ingested it —
    # every Bronze table in this course carries these two columns
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

print("Autoloader stream defined for customers.")
print(f"Will write to: {customers_target_table}")

In [ ]:
# ─── Step 2: Write customers to Bronze as a managed Delta table ───────────────
# trigger(availableNow=True) = process every file sitting there right now,
# then stop — this is the batch-style trigger, like a scheduled job that runs
# once per invocation (as opposed to a 24/7 streaming trigger).
# .toTable() registers the output as a real Unity Catalog table
# (gbmart.bronze.customers) — anyone with catalog access can query it by name,
# no ADLS path needed.

customers_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", customers_checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(customers_target_table)

# Verify
customers_count = spark.table(customers_target_table).count()
print(f"Customers in Bronze: {customers_count:,} rows")
print(f"Table: {customers_target_table}")

In [ ]:
# ─── Step 3: Inspect the result ──────────────────────────────────────────────
# Read it back as a normal table — no path, just the 3-level UC name

customers_bronze = spark.table(customers_target_table)

print("Schema inferred by Autoloader:")
customers_bronze.printSchema()

print("\nSample rows (with audit columns):")
customers_bronze.show(5, truncate=True)

print("\nAudit columns added:")
customers_bronze.select("_ingested_at", "_source_file").show(3, truncate=False)

---
## Part 2 — Ingest payments.csv with Autoloader

In [ ]:
# ─── Read + write payments/ using Autoloader — same pattern as customers ──────
# One Autoloader stream per source table is the production pattern: each
# table gets its own schema location, checkpoint, and target table.

payments_source_path     = f"{EXTERNAL_LOCATION}/payments/"
payments_schema_path     = f"{EXTERNAL_LOCATION}/_schemas/payments/"
payments_checkpoint_path = f"{EXTERNAL_LOCATION}/_checkpoints/payments/"
payments_target_table    = f"{CATALOG}.{SCHEMA}.payments"

payments_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",           "csv")
    .option("cloudFiles.schemaLocation",   payments_schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header",                      "true")
    .load(payments_source_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

payments_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", payments_checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(payments_target_table)

payments_count = spark.table(payments_target_table).count()
print(f"Payments in Bronze: {payments_count:,} rows")
print(f"Table: {payments_target_table}")

In [ ]:
# ─── Inspect payments ─────────────────────────────────────────────────────────

payments_bronze = spark.table(payments_target_table)

print("Payments schema:")
payments_bronze.printSchema()

print("\nSample rows:")
payments_bronze.show(5, truncate=True)

---
## Part 3 — The MAGIC of Autoloader: Run it again and see what happens

In [ ]:
# ─── Run Autoloader again on the SAME files ────────────────────────────────────
# Expectation: Autoloader checks its checkpoint, sees these files were already
# processed, and skips them — no duplicate rows land in Bronze.

print("Running Autoloader on customers/ again...")

customers_before = spark.table(customers_target_table).count()
print(f"Customers in Bronze BEFORE second run: {customers_before:,}")

# Re-define the same stream (schema/checkpoint paths are identical — that's
# what makes this a re-run of the SAME logical stream, not a new one)
customers_stream2 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",           "csv")
    .option("cloudFiles.schemaLocation",   customers_schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header",                      "true")
    .load(customers_source_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

customers_stream2.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", customers_checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(customers_target_table)

customers_after = spark.table(customers_target_table).count()
print(f"Customers in Bronze AFTER second run : {customers_after:,}")
print()

if customers_after == customers_before:
    print("File(s) already processed — Autoloader SKIPPED them!")
    print("No duplicates added.")
else:
    print(f"New rows added: {customers_after - customers_before:,} (a new file must have landed)")

In [ ]:
# ─── Check the checkpoint to understand HOW Autoloader tracks files ───────────
# The checkpoint folder contains metadata about every file that was processed.
# Autoloader checks this before processing — if a file is listed, it skips it.

print("Contents of the Autoloader checkpoint folder:")
dbutils.fs.ls(customers_checkpoint_path)

# You'll see folders like:
#   commits/         ← list of completed micro-batches
#   offsets/         ← tracks progress through the source
#   sources/         ← tracks which exact files have been read
#   metadata         ← query metadata

# This is why Autoloader never re-reads the same file:
# every file's path and size is recorded in the checkpoint after processing.

---
## Part 4 — Bonus: Read an entire folder (all sources at once)

> **Illustration only.** `raw-data/` also contains `orders/` and `order_items/` — in the real GlobalMart architecture those arrive via Postgres CDC (Lakeflow Connect, Day 2), not Autoloader. This cell reads the whole `raw-data/` folder anyway just to show that folder-level Autoloader reads are possible; it is not the pattern you'd use in production, where each source gets its own stream (as in Parts 1–2 above).

In [ ]:
# ─── Read ALL files under raw-data/ with one Autoloader stream ────────────────
# Instead of pointing to one source's subfolder, point at the whole
# raw-data/ root. Autoloader recurses and picks up every file underneath.
# NOTE: combining sources with DIFFERENT schemas into one table only "works"
# because addNewColumns absorbs the mismatch — it does NOT mean this is a
# good idea. This cell exists purely to show the folder-level read is
# possible; the real pipeline always uses one stream per source (Parts 1–2).

all_files_schema_path     = f"{EXTERNAL_LOCATION}/_schemas/all_raw_demo/"
all_files_checkpoint_path = f"{EXTERNAL_LOCATION}/_checkpoints/all_raw_demo/"
all_files_target_table    = f"{CATALOG}.{SCHEMA}.all_raw_combined_demo"

all_files_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",             "csv")
    .option("cloudFiles.schemaLocation",     all_files_schema_path)
    .option("cloudFiles.inferColumnTypes",   "true")
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")
    .option("header",                        "true")
    .load(EXTERNAL_LOCATION)  # <-- whole raw-data/ root, not one subfolder
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

all_files_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", all_files_checkpoint_path) \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(all_files_target_table)

total = spark.table(all_files_target_table).count()
print(f"Total rows from all sources combined: {total:,}")
print("(This demo table is disposable — drop it after the lab, it's not part of the real pipeline.)")

---
## Summary — Autoloader vs Manual CSV Read

| | Manual `spark.read.csv` | Autoloader (`cloudFiles`) |
|-|------------------------|---------------------------|
| **Tracks processed files** | No — reads everything every time | Yes — checkpoint remembers every file |
| **Handles new files** | You must add code to detect them | Automatic — detects new files on next run |
| **Schema inference** | Once per run | Inferred once, stored, reused across runs |
| **Schema evolution** | Breaks if a new column appears | `addNewColumns` mode handles it gracefully |
| **Streaming support** | No | Yes — can run as a continuous stream |
| **Duplicates risk** | High (re-reads on every run) | Zero (checkpoint prevents re-reads) |
| **When to use** | Quick one-off read | Production ingestion pipeline |

---
## What got created

```
Unity Catalog managed tables (queryable by name, no path needed):
  gbmart.bronze.customers            ← Autoloader output, Part 1
  gbmart.bronze.payments             ← Autoloader output, Part 2
  gbmart.bronze.all_raw_combined_demo ← bonus demo (Part 4, disposable)

External Location (source — read-only from this notebook):
  abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data/
    customers/       ← source CSVs
    payments/        ← source CSVs
    _schemas/        ← Autoloader's inferred schema, one folder per table
    _checkpoints/     ← Autoloader's "already processed" tracker, one per table
```

---
## Submission Checklist

```
Submission Checklist
────────────────────────────────────────────────────────
✅ Setup cell ran — "Connected via Unity Catalog External Location" printed
✅ customers ingested via Autoloader → gbmart.bronze.customers
── Customers Bronze row count:        ______
✅ payments ingested via Autoloader → gbmart.bronze.payments
── Payments Bronze row count:         ______
✅ Second run on customers produced ZERO new rows (checkpoint verified)
✅ Checkpoint folder inspected (commits/, offsets/, sources/, metadata)
✅ Bonus: folder-level Autoloader read attempted (Part 4)
────────────────────────────────────────────────────────
```